# CardioSafe AI - Notebook 03: deep-model benchmark

Trains five deep architectures (1D ResNet, xResNet, Inception, BiGRU, CNN-Transformer) with 3 seeds each, plus ablations (augmentation on/off, BCE vs focal loss), then compares everything against the baselines from notebook 02 and builds ensembles.

**Runtime: GPU (T4).** Needs notebook 02 done (`MyDrive/CardioSafe` with the data, `norm_stats.npz` and `results_02_baselines.csv`).

**Safe to interrupt:** every finished run saves its predictions and weights to Drive. If Colab disconnects, reconnect and Run all; finished runs are skipped. Total time is a couple of hours on a T4.

Selection rule: model types are chosen using **validation** results only. The test fold is used for reporting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
PRED = WORK + '/preds'; CKPT = WORK + '/models'
os.makedirs(PRED, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

### Updated shared modules (backwards compatible with notebook 02)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_common.py
"""Shared utilities for CardioSafe AI notebooks: data loading, evaluation, deep-model training."""
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


# ----------------------------------------------------------------- data
def load_data(path):
    z = np.load(path)
    X, Y, fold = z['X'], z['Y'], z['fold']
    tr, va, te = fold <= 8, fold == 9, fold == 10
    return X, Y, fold, tr, va, te, z


# ----------------------------------------------------------- evaluation
def tune_thresholds(y, p, grid=None):
    """Per-class threshold maximising F1 on the validation set."""
    grid = np.linspace(0.02, 0.98, 97) if grid is None else grid
    ths = []
    for c in range(y.shape[1]):
        f = [f1_score(y[:, c], p[:, c] >= t, zero_division=0) for t in grid]
        ths.append(grid[int(np.argmax(f))])
    return np.array(ths)


def macro_auc(y, p):
    return float(np.mean([roc_auc_score(y[:, c], p[:, c]) for c in range(y.shape[1])]))


def bootstrap_ci(y, p, n=500, seed=0, alpha=0.05):
    rng = np.random.default_rng(seed)
    N, vals = len(y), []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        try:
            vals.append(macro_auc(y[idx], p[idx]))
        except ValueError:      # a class missing in the resample
            continue
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def summarize(name, y_val, p_val, y_te, p_te, save_dir=None, ci=True):
    """Metrics on test, with thresholds tuned on validation. Optionally saves predictions."""
    ths = tune_thresholds(y_val, p_val)
    pred = p_te >= ths
    per_auc = [roc_auc_score(y_te[:, c], p_te[:, c]) for c in range(y_te.shape[1])]
    per_ap = [average_precision_score(y_te[:, c], p_te[:, c]) for c in range(y_te.shape[1])]
    per_f1 = [f1_score(y_te[:, c], pred[:, c], zero_division=0) for c in range(y_te.shape[1])]
    row = {'model': name, 'AUROC': np.mean(per_auc), 'AUPRC': np.mean(per_ap), 'F1': np.mean(per_f1)}
    if ci:
        lo, hi = bootstrap_ci(y_te, p_te)
        row['AUROC_lo'], row['AUROC_hi'] = lo, hi
    for c, n in enumerate(CLASSES):
        row[f'AUROC_{n}'] = per_auc[c]
    for c, n in enumerate(CLASSES):
        row[f'F1_{n}'] = per_f1[c]
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        np.savez(os.path.join(save_dir, f'{name}.npz'), val=p_val, test=p_te, thresholds=ths)
    return row


# --------------------------------------------------------- deep learning
def standardize_fit(X_train):
    """Per-lead mean/std from the training set. X shape (N, T, 12)."""
    mu = X_train.mean(axis=(0, 1), keepdims=True)
    sd = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)


def to_tensor_ncl(X, mu, sd, device):
    """(N, T, 12) numpy -> (N, 12, T) float tensor on device, standardised."""
    Xn = ((X - mu) / sd).transpose(0, 2, 1).astype(np.float32)
    return torch.from_numpy(Xn).to(device)


def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, Xt, bs=512):
    model.eval()
    out = []
    for i in range(0, len(Xt), bs):
        out.append(torch.sigmoid(model(Xt[i:i + bs]).float()).cpu())
    return torch.cat(out).numpy()


def train_deep(model, Xtr, Ytr, Xva, Yva, epochs=40, lr=2e-3, wd=1e-2, bs=128,
               seed=0, augment=None, loss_fn=None, clip=None, verbose=True):
    """Train with AdamW + OneCycle + AMP; keep the epoch with best validation macro AUROC.
    Xtr/Xva: tensors (N,12,T) on device. Ytr: float tensor (N,C) on device. Yva: numpy (N,C)."""
    set_seed(seed)
    device = Xtr.device
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = int(np.ceil(len(Xtr) / bs)) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.2)
    loss_fn = loss_fn or nn.BCEWithLogitsLoss()
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        perm = torch.randperm(len(Xtr), device=device)
        tot = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            xb, yb = Xtr[idx], Ytr[idx]
            if augment is not None:
                xb = augment(xb)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = loss_fn(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if clip:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        pv = predict(model, Xva)
        auc = macro_auc(Yva, pv)
        hist.append((ep, tot / len(Xtr), auc))
        if auc > best:
            best, best_state = auc, copy.deepcopy(model.state_dict())
        if verbose:
            print(f'ep {ep + 1:02d}/{epochs}  loss {tot / len(Xtr):.4f}  val AUROC {auc:.4f}  best {best:.4f}  ({time.time() - t0:.0f}s)')
    model.load_state_dict(best_state)
    return model, hist


# ----------------------------------------------------------- baseline CNN
def make_cnn1d(n_classes=5, in_ch=12, width=32, p_drop=0.3):
    class GlobalPool(nn.Module):
        def forward(self, x):
            return torch.cat([x.mean(-1), x.amax(-1)], dim=1)

    def block(i, o, k):
        return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False), nn.BatchNorm1d(o),
                             nn.ReLU(inplace=True), nn.MaxPool1d(2))

    w = width
    return nn.Sequential(
        block(in_ch, w, 15), block(w, 2 * w, 9), block(2 * w, 4 * w, 7), block(4 * w, 8 * w, 5),
        GlobalPool(), nn.Dropout(p_drop), nn.Linear(16 * w, n_classes))


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_models.py
"""Model zoo and augmentation for CardioSafe AI. All models: input (B,12,T) -> logits (B,C)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as Fn


class Pool(nn.Module):
    """Global average + max pooling over time -> (B, 2*C)."""
    def forward(self, x):
        return torch.cat([x.mean(-1), x.amax(-1)], dim=1)


# ------------------------------------------------------------------ ResNet
class BasicBlock(nn.Module):
    def __init__(self, i, o, stride=1, k=7, drop=0.0):
        super().__init__()
        self.c1 = nn.Conv1d(i, o, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(o)
        self.c2 = nn.Conv1d(o, o, k, 1, k // 2, bias=False); self.b2 = nn.BatchNorm1d(o)
        self.drop = nn.Dropout(drop) if drop > 0 else nn.Identity()
        self.short = None
        if stride != 1 or i != o:
            self.short = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))

    def forward(self, x):
        y = Fn.relu(self.b1(self.c1(x)), inplace=True)
        y = self.b2(self.c2(self.drop(y)))
        return Fn.relu(y + (x if self.short is None else self.short(x)), inplace=True)


class ResNet1D(nn.Module):
    """ResNet-18 style 1D network (our main model)."""
    def __init__(self, n_classes=5, in_ch=12, width=64, layers=(2, 2, 2, 2), k=7, p_drop=0.3, block_drop=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, width, 15, 2, 7, bias=False), nn.BatchNorm1d(width),
                                  nn.ReLU(inplace=True), nn.MaxPool1d(3, 2, 1))
        blocks, c = [], width
        for li, n in enumerate(layers):
            o = width * 2 ** li
            for bi in range(n):
                blocks.append(BasicBlock(c, o, stride=2 if (bi == 0 and li > 0) else 1, k=k, drop=block_drop))
                c = o
        self.body = nn.Sequential(*blocks)
        self.head = nn.Sequential(Pool(), nn.Dropout(p_drop), nn.Linear(2 * c, n_classes))

    def forward(self, x):
        return self.head(self.body(self.stem(x)))


# ---------------------------------------------------------------- xResNet
class XBlock(nn.Module):
    """ResNet-D style block: avg-pool + 1x1 conv shortcut, zero-init last BN gamma."""
    def __init__(self, i, o, stride=1, k=5):
        super().__init__()
        self.c1 = nn.Conv1d(i, o, k, 1, k // 2, bias=False); self.b1 = nn.BatchNorm1d(o)
        self.c2 = nn.Conv1d(o, o, k, stride, k // 2, bias=False); self.b2 = nn.BatchNorm1d(o)
        nn.init.zeros_(self.b2.weight)
        sc = []
        if stride > 1:
            sc.append(nn.AvgPool1d(stride, stride, ceil_mode=True))
        if i != o:
            sc += [nn.Conv1d(i, o, 1, bias=False), nn.BatchNorm1d(o)]
        self.short = nn.Sequential(*sc) if sc else nn.Identity()

    def forward(self, x):
        y = self.b2(self.c2(Fn.relu(self.b1(self.c1(x)), inplace=True)))
        return Fn.relu(y + self.short(x), inplace=True)


class XResNet1D(nn.Module):
    def __init__(self, n_classes=5, in_ch=12, width=32, layers=(2, 2, 2, 2), p_drop=0.3):
        super().__init__()
        w = width
        self.stem = nn.Sequential(
            nn.Conv1d(in_ch, w, 5, 2, 2, bias=False), nn.BatchNorm1d(w), nn.ReLU(inplace=True),
            nn.Conv1d(w, w, 5, 1, 2, bias=False), nn.BatchNorm1d(w), nn.ReLU(inplace=True),
            nn.Conv1d(w, 2 * w, 5, 1, 2, bias=False), nn.BatchNorm1d(2 * w), nn.ReLU(inplace=True),
            nn.MaxPool1d(3, 2, 1))
        blocks, c = [], 2 * w
        for li, n in enumerate(layers):
            o = 2 * w * 2 ** li
            for bi in range(n):
                blocks.append(XBlock(c, o, stride=2 if (bi == 0 and li > 0) else 1)); c = o
        self.body = nn.Sequential(*blocks)
        self.head = nn.Sequential(Pool(), nn.Dropout(p_drop), nn.Linear(2 * c, n_classes))

    def forward(self, x):
        return self.head(self.body(self.stem(x)))


# -------------------------------------------------------------- Inception
class InceptionModule(nn.Module):
    def __init__(self, i, nf=32, ks=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bott = nn.Conv1d(i, bottleneck, 1, bias=False) if i > 1 else nn.Identity()
        bi = bottleneck if i > 1 else i
        self.convs = nn.ModuleList([nn.Conv1d(bi, nf, k, 1, k // 2, bias=False) for k in ks])
        self.mp = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(i, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))

    def forward(self, x):
        z = self.bott(x)
        out = torch.cat([c(z) for c in self.convs] + [self.mp(x)], dim=1)
        return Fn.relu(self.bn(out), inplace=True)


class Inception1D(nn.Module):
    def __init__(self, n_classes=5, in_ch=12, nf=32, depth=6, p_drop=0.3):
        super().__init__()
        self.down = nn.AvgPool1d(2)                      # 1000 -> 500 samples for speed
        out_ch = nf * 4
        self.mods = nn.ModuleList([InceptionModule(in_ch if d == 0 else out_ch, nf) for d in range(depth)])
        self.res = nn.ModuleList([nn.Sequential(nn.Conv1d(in_ch if d == 0 else out_ch, out_ch, 1, bias=False),
                                                nn.BatchNorm1d(out_ch)) for d in range(0, depth, 3)])
        self.head = nn.Sequential(Pool(), nn.Dropout(p_drop), nn.Linear(2 * out_ch, n_classes))

    def forward(self, x):
        x = self.down(x)
        r = x
        for d, m in enumerate(self.mods):
            x = m(x)
            if d % 3 == 2:
                x = Fn.relu(x + self.res[d // 3](r), inplace=True)
                r = x
        return self.head(x)


# --------------------------------------------------------- BiGRU / hybrid
class ConvStem(nn.Module):
    def __init__(self, in_ch, out_ch, down):
        super().__init__()
        layers, c = [], in_ch
        for j in range(down):
            o = out_ch if j == down - 1 else out_ch // 2
            layers += [nn.Conv1d(c, o, 7, 2, 3, bias=False), nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
            c = o
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class BiGRU(nn.Module):
    def __init__(self, n_classes=5, in_ch=12, hidden=128, p_drop=0.3):
        super().__init__()
        self.stem = ConvStem(in_ch, 64, 2)               # 1000 -> 250 steps
        self.rnn = nn.GRU(64, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.drop = nn.Dropout(p_drop); self.fc = nn.Linear(4 * hidden, n_classes)

    def forward(self, x):
        h, _ = self.rnn(self.stem(x).transpose(1, 2))
        return self.fc(self.drop(torch.cat([h.mean(1), h.amax(1)], dim=1)))


class CNNTransformer(nn.Module):
    def __init__(self, n_classes=5, in_ch=12, d=128, heads=4, layers=3, ff=256, p_drop=0.3, max_len=512):
        super().__init__()
        self.stem = ConvStem(in_ch, d, 3)                # 1000 -> 125 tokens
        self.pos = nn.Parameter(torch.zeros(1, max_len, d)); nn.init.trunc_normal_(self.pos, std=0.02)
        enc = nn.TransformerEncoderLayer(d, heads, ff, dropout=0.1, batch_first=True, norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(enc, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(p_drop); self.fc = nn.Linear(2 * d, n_classes)

    def forward(self, x):
        t = self.stem(x).transpose(1, 2)
        t = self.enc(t + self.pos[:, :t.shape[1]])
        t = self.norm(t)
        return self.fc(self.drop(torch.cat([t.mean(1), t.amax(1)], dim=1)))


REGISTRY = {
    'resnet1d':    lambda: ResNet1D(),
    'xresnet1d':   lambda: XResNet1D(),
    'inception1d': lambda: Inception1D(),
    'bigru':       lambda: BiGRU(),
    'cnn_transformer': lambda: CNNTransformer(),
}


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# ------------------------------------------------------------ augmentation
def make_augment(p_scale=0.8, scale=(0.8, 1.2), noise=0.03, max_shift=50, p_wander=0.5, wander_amp=0.15, fs=100):
    """GPU batch augmentation on standardised signals (B,12,T): amplitude scale, noise, circular time shift, baseline wander."""
    def aug(x):
        B, L, T = x.shape
        dev = x.device
        s = torch.empty(B, 1, 1, device=dev).uniform_(*scale)
        x = torch.where(torch.rand(B, 1, 1, device=dev) < p_scale, x * s, x)
        x = x + noise * torch.randn_like(x)
        if max_shift:
            sh = int(torch.randint(-max_shift, max_shift + 1, (1,)).item())
            x = torch.roll(x, sh, dims=-1)
        f = torch.empty(B, 1, 1, device=dev).uniform_(0.05, 0.5)
        ph = torch.empty(B, 1, 1, device=dev).uniform_(0, 2 * math.pi)
        tt = torch.arange(T, device=dev).view(1, 1, T) / fs
        w = wander_amp * torch.sin(2 * math.pi * f * tt + ph)
        return x + torch.where(torch.rand(B, 1, 1, device=dev) < p_wander, w, torch.zeros_like(w))
    return aug


# --------------------------------------------------------------- losses
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma

    def forward(self, logits, y):
        bce = Fn.binary_cross_entropy_with_logits(logits, y, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * y + (1 - p) * (1 - y)
        return ((1 - pt) ** self.gamma * bce).mean()


In [ ]:
import importlib, time, numpy as np, pandas as pd, torch
import cardiosafe_common as C, cardiosafe_models as MZ
importlib.reload(C); importlib.reload(MZ)

X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
dev = torch.device('cuda')
mu, sd = C.standardize_fit(X[tr])
Xtr, Xva, Xte = [C.to_tensor_ncl(X[m], mu, sd, dev) for m in (tr, va, te)]
Ytr = torch.tensor(Y[tr]).to(dev)
print('train/val/test', len(Xtr), len(Xva), len(Xte), '|', torch.cuda.get_device_name(0))
for k, f in MZ.REGISTRY.items():
    print(f'{k:16s} {MZ.n_params(f()) / 1e6:.2f}M parameters')

## Experiment plan
Main models run with 3 seeds. Ablations use 1 seed (a difference smaller than the seed-to-seed spread, about 0.002 AUROC, is not meaningful).

In [ ]:
AUG = MZ.make_augment()
# name -> (architecture, augmentation, loss, lr, seeds)
EXPERIMENTS = {
    'resnet1d':        ('resnet1d',        True,  'bce',   2e-3, [0, 1, 2]),
    'xresnet1d':       ('xresnet1d',       True,  'bce',   2e-3, [0, 1, 2]),
    'inception1d':     ('inception1d',     True,  'bce',   2e-3, [0, 1, 2]),
    'bigru':           ('bigru',           True,  'bce',   2e-3, [0, 1, 2]),
    'cnn_transformer': ('cnn_transformer', True,  'bce',   1e-3, [0, 1, 2]),
    # ablations (1 seed)
    'resnet1d_noaug':  ('resnet1d',        False, 'bce',   2e-3, [0]),
    'resnet1d_focal':  ('resnet1d',        True,  'focal', 2e-3, [0]),
}
EPOCHS = 40
rows = {}      # run name -> result row

def run_one(exp, arch, aug, loss, lr, seed):
    run = f'{exp}_s{seed}'
    pfile = f'{PRED}/{run}.npz'
    if os.path.exists(pfile):
        z = np.load(pfile); pv, pt = z['val'], z['test']
        print(f'[skip] {run} (already done)')
    else:
        print(f'===== {run} =====')
        net = MZ.REGISTRY[arch]()
        t0 = time.time()
        net, hist = C.train_deep(net, Xtr, Ytr, Xva, Y[va], epochs=EPOCHS, lr=lr, bs=128, seed=seed,
                                 augment=AUG if aug else None,
                                 loss_fn=MZ.FocalBCE() if loss == 'focal' else None, clip=1.0, verbose=False)
        pv, pt = C.predict(net, Xva), C.predict(net, Xte)
        torch.save(net.state_dict(), f'{CKPT}/{run}.pt')
        print(f'   done in {(time.time() - t0) / 60:.1f} min, best val AUROC {max(h[2] for h in hist):.4f}')
    row = C.summarize(run, Y[va], pv, Y[te], pt, save_dir=PRED)
    row.update(exp=exp, seed=seed, val_AUROC=C.macro_auc(Y[va], pv))
    rows[run] = row
    print(f"   test AUROC {row['AUROC']:.4f}  F1 {row['F1']:.4f}")

## Run all experiments (long)

In [ ]:
for exp, (arch, aug, loss, lr, seeds) in EXPERIMENTS.items():
    for s in seeds:
        run_one(exp, arch, aug, loss, lr, s)
pd.DataFrame(rows).T.to_csv(WORK + '/results_03_runs.csv')

## Aggregate: mean ± std over seeds, per model

In [ ]:
df = pd.DataFrame(rows).T
for c in ['AUROC', 'AUPRC', 'F1', 'val_AUROC'] + [f'AUROC_{k}' for k in C.CLASSES]:
    df[c] = df[c].astype(float)
agg = df.groupby('exp').agg(AUROC_mean=('AUROC', 'mean'), AUROC_std=('AUROC', 'std'), AUPRC=('AUPRC', 'mean'),
                            F1=('F1', 'mean'), val_AUROC=('val_AUROC', 'mean'), n_seeds=('seed', 'count'))
# seed-averaged ensemble per model type
ens_rows = []
for exp in df.exp.unique():
    runs = df[df.exp == exp].index
    if len(runs) < 2: continue
    pv = np.mean([np.load(f'{PRED}/{r}.npz')['val'] for r in runs], 0)
    pt = np.mean([np.load(f'{PRED}/{r}.npz')['test'] for r in runs], 0)
    r = C.summarize(f'{exp}_seedavg', Y[va], pv, Y[te], pt, save_dir=PRED)
    ens_rows.append(r)
    agg.loc[exp, 'seedavg_AUROC'] = r['AUROC']
display(agg.sort_values('AUROC_mean', ascending=False).round(4))

## Compare with the notebook 02 baselines and build ensembles
Ensemble members are the seed-averaged predictions of the main model types. "top3" picks the 3 best model types by **validation** AUROC.

In [ ]:
main = [e for e, v in EXPERIMENTS.items() if len(v[4]) == 3]
P = {e: (np.load(f'{PRED}/{e}_seedavg.npz')['val'], np.load(f'{PRED}/{e}_seedavg.npz')['test']) for e in main}
val_rank = sorted(main, key=lambda e: -C.macro_auc(Y[va], P[e][0]))
print('validation ranking of model types:', [(e, round(C.macro_auc(Y[va], P[e][0]), 4)) for e in val_rank])
top3 = val_rank[:3]
ens = {
    'ensemble_all_deep': main,
    'ensemble_top3_by_val': top3,
}
final = []
for name, members in ens.items():
    pv = np.mean([P[m][0] for m in members], 0); pt = np.mean([P[m][1] for m in members], 0)
    r = C.summarize(name, Y[va], pv, Y[te], pt, save_dir=PRED)
    r['members'] = ','.join(members); final.append(r)
    print(f"{name:22s} AUROC {r['AUROC']:.4f} [{r['AUROC_lo']:.4f}, {r['AUROC_hi']:.4f}]  members: {members}")

base = pd.read_csv(WORK + '/results_02_baselines.csv', index_col='model')
base = base.loc[['logreg', 'decision_tree', 'random_forest', 'xgboost', 'cnn1d_3seed_mean']]
deep = pd.DataFrame([r for r in ens_rows if r['model'].replace('_seedavg', '') in main]).set_index('model')
allm = pd.concat([base, deep, pd.DataFrame(final).set_index('model')])
cols = ['AUROC', 'AUROC_lo', 'AUROC_hi', 'AUPRC', 'F1'] + [f'AUROC_{k}' for k in C.CLASSES]
allm = allm[cols].sort_values('AUROC', ascending=False)
allm.to_csv(WORK + '/results_03_final_comparison.csv')
display(allm.round(4))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
plot = allm.sort_values('AUROC')
ax.barh(plot.index, plot['AUROC'], xerr=[plot['AUROC'] - plot['AUROC_lo'], plot['AUROC_hi'] - plot['AUROC']], color='#3b7ddd', capsize=3)
ax.set_xlim(0.7, 0.95); ax.set_xlabel('Test macro AUROC (95% bootstrap CI)'); ax.set_title('PTB-XL superclass benchmark')
plt.tight_layout(); plt.savefig(WORK + '/fig_benchmark.png', dpi=150); plt.show()

## Next
Send me the aggregate table, the final comparison table, and the validation ranking line. Then notebook 04: calibration (temperature scaling, ECE, reliability diagrams), uncertainty (MC dropout, deep ensembles, conformal prediction) and explainability (Grad-CAM with a faithfulness test).